# SW-KAN 3D Function Approximation

Benchmarks an SW-KAN on multiple 3D function-approximation tasks using 5,000 samples and 5,000 epochs.

In [ ]:
"""
sw_kan_3d_function_approximation_5k_5000ep.py
===============================================
3D function approximation with SW-KAN on 5k training samples, 5000 epochs.
Run: python3 sw_kan_3d_function_approximation_5k_5000ep.py
"""

import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

torch.manual_seed(42)
np.random.seed(42)


# ============================================================================ #
# 1. SW-KAN MODEL
# ============================================================================ #

import math

class StieltjesWigertKANLayer(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        degree: int = 6,
        gamma: float = 2.0,
        iota: float = 1.0,
        q_min: float = 0.35,
        q_max: float = 0.95,
        base_scale: float = 1.0,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.degree = degree
        self.gamma = gamma
        self.iota = iota
        self.q_min = q_min
        self.q_max = q_max

        self.pre_scale = nn.Parameter(torch.ones(in_features))
        self.pre_shift = nn.Parameter(torch.zeros(in_features))

        fan_in = in_features * (degree + 1)
        self.coeffs = nn.Parameter(
            torch.randn(out_features, in_features, degree + 1) / math.sqrt(fan_in)
        )

        self.q_raw = nn.Parameter(torch.zeros(out_features, in_features))
        self.base_weight = nn.Parameter(
            torch.full((out_features, in_features), base_scale) / math.sqrt(in_features)
        )
        self.bias = nn.Parameter(torch.zeros(out_features))

    def domain_map(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pre_scale * x + self.pre_shift
        return torch.exp(self.gamma * torch.tanh(x / self.iota))

    def edge_q(self) -> torch.Tensor:
        s = torch.sigmoid(self.q_raw)
        return self.q_min + (self.q_max - self.q_min) * s

    def sw_basis(self, z: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
        batch = z.shape[0]
        device, dtype = z.device, z.dtype
        eps = 1e-8

        z_b = z.unsqueeze(1)
        log_q = torch.log(q).unsqueeze(0)
        q_val = torch.exp(log_q)

        def b_n(n: int) -> torch.Tensor:
            return torch.exp(-(2 * n + 1.5) * log_q) * (1.0 + q_val - torch.exp((n + 1) * log_q))

        def a2_n(n: int) -> torch.Tensor:
            if n == 0:
                return torch.zeros_like(q_val)
            return torch.exp(-4 * n * log_q) * (1.0 - torch.exp(n * log_q))

        T_prev = torch.zeros(batch, self.out_features, self.in_features, device=device, dtype=dtype)
        T_curr = torch.ones(batch, self.out_features, self.in_features, device=device, dtype=dtype)
        T_list = [T_curr]

        a_curr = torch.zeros(1, self.out_features, self.in_features, device=device, dtype=dtype)

        for n in range(self.degree):
            a_next = torch.sqrt(a2_n(n + 1) + eps)
            bn = b_n(n)
            T_next = ((z_b - bn) * T_curr - a_curr * T_prev) / a_next
            T_list.append(T_next)
            T_prev, T_curr = T_curr, T_next
            a_curr = a_next

        return torch.stack(T_list, dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.domain_map(x)
        q = self.edge_q()
        S = self.sw_basis(z, q)

        poly = torch.einsum("boin,oin->bo", S, self.coeffs)
        base = F.silu(x) @ self.base_weight.t()

        return poly + base + self.bias


class SWKAN(nn.Module):
    def __init__(self, layer_sizes: list[int], degree: int = 6, gamma: float = 2.0, iota: float = 1.0, q_min: float = 0.35, q_max: float = 0.95):
        super().__init__()
        assert len(layer_sizes) >= 2
        self.layers = nn.ModuleList(
            [
                StieltjesWigertKANLayer(
                    layer_sizes[i],
                    layer_sizes[i + 1],
                    degree=degree,
                    gamma=gamma,
                    iota=iota,
                    q_min=q_min,
                    q_max=q_max,
                )
                for i in range(len(layer_sizes) - 1)
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


# ============================================================================ #
# 2. TARGET FUNCTIONS
# ============================================================================ #

def function_1(x, y):
    return torch.sin(x) * torch.cos(y) + 0.5 * x * y

def function_2(x, y):
    return torch.exp(-(x**2 + y**2) / 2) * (1 + 0.3 * torch.sin(3*x) * torch.cos(2*y))

def function_3(x, y):
    return x**2 + y**2 + 0.5 * torch.sin(4*x) + 0.3 * torch.cos(4*y)


FUNCTIONS = [
    ("f₁ = sin(x)cos(y) + 0.5xy", function_1),
    ("f₂ = exp(-(x²+y²)/2)(1+0.3sin(3x)cos(2y))", function_2),
    ("f₃ = x²+y²+0.5sin(4x)+0.3cos(4y)", function_3),
]


# ============================================================================ #
# 3. TRAINING
# ============================================================================ #

def train_model(model, X_train, y_train, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()
    
    t0 = time.time()
    losses = []
    
    for epoch in range(epochs):
        opt.zero_grad()
        pred = model(X_train)
        loss = loss_fn(pred, y_train)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        sched.step()
        
        losses.append(loss.item())
        
        if epoch % 500 == 0 or epoch == epochs - 1:
            print(f"    epoch {epoch:5d}  loss: {loss.item():.6f}")
    
    elapsed = time.time() - t0
    return losses, elapsed


# ============================================================================ #
# 4. MAIN - 5k SAMPLES, 5000 EPOCHS, FAST PLOTTING
# ============================================================================ #

if __name__ == "__main__":
    print("=" * 70)
    print("SW-KAN 3D Function Approximation (5k samples, 5000 epochs)")
    print("=" * 70)
    
    # Parameters - 5k samples, 5000 epochs
    n_train = 5000
    n_test = 25  # کاهش برای سرعت بیشتر
    epochs = 5000
    lr = 1e-2
    
    print(f"\nTraining configuration:")
    print(f"  - Training samples: {n_train}")
    print(f"  - Epochs: {epochs}")
    print(f"  - Learning rate: {lr}")
    print(f"  - Network: [2, 16, 1], degree=4")
    print(f"  - Test grid: {n_test}x{n_test} = {n_test*n_test} points")
    
    # Training data
    X_train = torch.rand(n_train, 2) * 4 - 2
    
    # Test grid
    x_test = torch.linspace(-2, 2, n_test)
    y_test = torch.linspace(-2, 2, n_test)
    X1_test, X2_test = torch.meshgrid(x_test, y_test, indexing='ij')
    X_test_grid = torch.stack([X1_test.flatten(), X2_test.flatten()], dim=1)
    
    results = []
    
    # Train for each function
    for func_name, func in FUNCTIONS:
        print("\n" + "-" * 70)
        print(f"Training: {func_name}")
        print("-" * 70)
        
        y_train = func(X_train[:, 0], X_train[:, 1]).unsqueeze(1)
        y_test_true = func(X1_test, X2_test)
        
        model = SWKAN([2, 16, 1], degree=4)
        
        losses, elapsed = train_model(model, X_train, y_train, epochs=epochs, lr=lr)
        
        with torch.no_grad():
            pred = model(X_test_grid).reshape(n_test, n_test)
        
        test_mse = torch.mean((pred - y_test_true)**2).item()
        
        results.append({
            'name': func_name,
            'model': model,
            'pred': pred,
            'y_test': y_test_true,
            'losses': losses,
            'test_mse': test_mse,
            'elapsed': elapsed,
            'params': sum(p.numel() for p in model.parameters() if p.requires_grad)
        })
        
        print(f"    Test MSE: {test_mse:.6f}")
        print(f"    Time: {elapsed:.2f}s")
        print(f"    Parameters: {results[-1]['params']:,}")

    # ======================================================================== #
    # 5. PLOT - FAST (reduced resolution)
    # ======================================================================== #
    
    print("\nGenerating plots (fast mode)...")
    
    # ایجاد یک شکل با ساب‌پلات‌های ۳x۳
    fig = plt.figure(figsize=(15, 10))
    
    X_plot, Y_plot = X1_test, X2_test
    
    for idx, data in enumerate(results):
        # Ground Truth
        ax1 = fig.add_subplot(3, 3, idx*3 + 1, projection='3d')
        ax1.plot_surface(
            X_plot.numpy(), Y_plot.numpy(), data['y_test'].numpy(),
            cmap='viridis', alpha=0.8, linewidth=0, antialiased=True,
            rstride=1, cstride=1
        )
        ax1.set_title(f"Ground Truth", fontsize=9)
        ax1.set_xlabel('x', fontsize=7)
        ax1.set_ylabel('y', fontsize=7)
        ax1.set_zlabel('f', fontsize=7)
        ax1.view_init(elev=25, azim=45)
        ax1.tick_params(labelsize=6)
        
        # Prediction
        ax2 = fig.add_subplot(3, 3, idx*3 + 2, projection='3d')
        ax2.plot_surface(
            X_plot.numpy(), Y_plot.numpy(), data['pred'].numpy(),
            cmap='plasma', alpha=0.8, linewidth=0, antialiased=True,
            rstride=1, cstride=1
        )
        ax2.set_title(f"SW-KAN Prediction", fontsize=9)
        ax2.set_xlabel('x', fontsize=7)
        ax2.set_ylabel('y', fontsize=7)
        ax2.set_zlabel('f', fontsize=7)
        ax2.view_init(elev=25, azim=45)
        ax2.tick_params(labelsize=6)
        
        # Error
        ax3 = fig.add_subplot(3, 3, idx*3 + 3, projection='3d')
        error = torch.abs(data['pred'] - data['y_test'])
        ax3.plot_surface(
            X_plot.numpy(), Y_plot.numpy(), error.numpy(),
            cmap='Reds', alpha=0.8, linewidth=0, antialiased=True,
            rstride=1, cstride=1
        )
        ax3.set_title(f"Absolute Error", fontsize=9)
        ax3.set_xlabel('x', fontsize=7)
        ax3.set_ylabel('y', fontsize=7)
        ax3.set_zlabel('|error|', fontsize=7)
        ax3.view_init(elev=25, azim=45)
        ax3.tick_params(labelsize=6)
    
    fig.suptitle("SW-KAN 3D Function Approximation (5k samples, 5000 epochs)", fontsize=13, y=1.02)
    plt.tight_layout()
    
    # ======================================================================== #
    # 6. LOSS CURVES
    # ======================================================================== #
    
    fig_loss = plt.figure(figsize=(8, 4))
    ax_loss = fig_loss.add_subplot(111)
    for data in results:
        step = max(1, len(data['losses']) // 80)
        losses = data['losses'][::step]
        ax_loss.plot(range(0, len(data['losses']), step), losses, label=data['name'][:35], linewidth=1.5)
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Training Loss (MSE)')
    ax_loss.set_title('Training Loss Curves (5k samples, 5000 epochs)')
    ax_loss.legend(fontsize=8)
    ax_loss.grid(True, alpha=0.3)
    ax_loss.set_yscale('log')
    
    plt.tight_layout()
    
    # Display
    try:
        from IPython.display import display
        display(fig)
        display(fig_loss)
        print("\nPlots displayed inline.")
    except:
        fig.savefig("sw_kan_3d_functions_5k_5000ep.png", dpi=100, bbox_inches="tight")
        fig_loss.savefig("sw_kan_training_losses_5k_5000ep.png", dpi=100, bbox_inches="tight")
        print("\nSaved plots to sw_kan_3d_functions_5k_5000ep.png and sw_kan_training_losses_5k_5000ep.png")
    
    plt.show()
    
    # ======================================================================== #
    # 7. SUMMARY
    # ======================================================================== #
    
    print("\n" + "=" * 70)
    print("SUMMARY (5k samples, 5000 epochs)")
    print("=" * 70)
    print(f"{'Function':<50} | {'Test MSE':>12} | {'Params':>10} | {'Time (s)':>10}")
    print("-" * 70)
    for data in results:
        print(f"{data['name'][:50]:<50} | {data['test_mse']:>12.6f} | {data['params']:>10,} | {data['elapsed']:>10.2f}")